In [1]:
import math
import random

class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Inicializar pesos y biases con valores aleatorios pequeños
        # Pesos entre capa de entrada y capa oculta (matriz input_size x hidden_size)
        self.weights1 = [[random.uniform(-0.1, 0.1) for _ in range(hidden_size)] for _ in range(input_size)]
        # Biases para la capa oculta (vector de tamaño hidden_size)
        self.biases1 = [random.uniform(-0.1, 0.1) for _ in range(hidden_size)]
        # Pesos entre capa oculta y capa de salida (matriz hidden_size x output_size)
        self.weights2 = [[random.uniform(-0.1, 0.1) for _ in range(output_size)] for _ in range(hidden_size)]
        # Biases para la capa de salida (vector de tamaño output_size)
        self.biases2 = [random.uniform(-0.1, 0.1) for _ in range(output_size)]
    
    @staticmethod
    def relu(x):
        return max(0, x)
    
    @staticmethod
    def relu_derivative(x):
        return 1 if x > 0 else 0
    
    @staticmethod
    def softmax(x):
        exp_values = [math.exp(i) for i in x]  # Exponenciación de cada elemento
        sum_exp = sum(exp_values)  # Suma de todos los exponenciales
        return [i/sum_exp for i in exp_values]  # Normalización
    
    def forward(self, x):
        # Capa oculta
        self.hidden = []
        for i in range(self.hidden_size):
            # Calcula la entrada ponderada para cada neurona oculta
            neuron_input = sum(x[j] * self.weights1[j][i] for j in range(self.input_size))
            # Aplica ReLU a la suma de entradas ponderadas + bias
            self.hidden.append(self.relu(neuron_input + self.biases1[i]))
        
        # Capa de salida
        self.output = []
        for i in range(self.output_size):
            # Calcula la entrada ponderada para cada neurona de salida
            neuron_input = sum(self.hidden[j] * self.weights2[j][i] for j in range(self.hidden_size))
            # Suma las entradas ponderadas + bias (sin función de activación aquí)
            self.output.append(neuron_input + self.biases2[i])
        
        # Aplica softmax a la salida para obtener probabilidades
        return self.softmax(self.output)
    
    def train(self, X, y, epochs=1000, learning_rate=0.01):
        for epoch in range(epochs):
            total_loss = 0
            for i in range(len(X)):
                # Forward pass: calcula la salida de la red
                output = self.forward(X[i])
                
                # Calcular pérdida (entropía cruzada)
                total_loss += -sum(y[i][j] * math.log(output[j] + 1e-10) for j in range(self.output_size))
                
                # Backpropagation
                # Gradiente capa salida (error en la capa de salida)
                output_error = [output[j] - y[i][j] for j in range(self.output_size)]
                
                # Gradiente capa oculta (propagación del error hacia atrás)
                hidden_error = []
                for h in range(self.hidden_size):
                    error = sum(output_error[j] * self.weights2[h][j] for j in range(self.output_size))
                    hidden_error.append(error * self.relu_derivative(self.hidden[h]))
                
                # Actualizar pesos y biases (descenso de gradiente)
                # Capa oculta -> capa salida
                for h in range(self.hidden_size):
                    for o in range(self.output_size):
                        self.weights2[h][o] -= learning_rate * output_error[o] * self.hidden[h]
                
                # Biases capa salida
                for o in range(self.output_size):
                    self.biases2[o] -= learning_rate * output_error[o]
                
                # Capa entrada -> capa oculta
                for i_ in range(self.input_size):
                    for h in range(self.hidden_size):
                        self.weights1[i_][h] -= learning_rate * hidden_error[h] * X[i][i_]
                
                # Biases capa oculta
                for h in range(self.hidden_size):
                    self.biases1[h] -= learning_rate * hidden_error[h]
            
            # Mostrar progreso cada 100 épocas
            if epoch % 100 == 0:
                print(f"Época {epoch}, Pérdida: {total_loss/len(X)}")
    
    def predict(self, x):
        output = self.forward(x)
        return output.index(max(output))  # Devuelve el índice del valor máximo



In [2]:
# Dataset de entrenamiento: todas las combinaciones posibles de 4 bits
dataset = [
    [0,0,0,0], [0,0,0,1], [0,0,1,0], [0,0,1,1],
    [0,1,0,0], [0,1,0,1], [0,1,1,0], [0,1,1,1],
    [1,0,0,0], [1,0,0,1], [1,0,1,0], [1,0,1,1],
    [1,1,0,0], [1,1,0,1], [1,1,1,0], [1,1,1,1]
]

# Etiquetas: one-hot encoding para 16 clases (0-15)
labels = [[1 if i == j else 0 for j in range(16)] for i in range(16)]

# Crear y entrenar la red neuronal
print("Entrenando la red...")
nn = NeuralNetwork(input_size=4, hidden_size=8, output_size=16)
nn.train(dataset, labels, epochs=1000, learning_rate=0.01)

# Probar la red con todas las entradas posibles
hex_digits = "0123456789ABCDEF"  # Para mostrar resultados en hexadecimal
print("\nResultados:")
for i, data in enumerate(dataset):
    pred = nn.predict(data)
    print(f"Entrada: {data} -> Pred: {hex_digits[pred]}, Real: {hex_digits[i]}")

Entrenando la red...
Época 0, Pérdida: 2.7793247310039297
Época 100, Pérdida: 2.618356327973826
Época 200, Pérdida: 1.6230718088452658
Época 300, Pérdida: 0.7632490159584266
Época 400, Pérdida: 0.2526814265326501
Época 500, Pérdida: 0.11174027415915021
Época 600, Pérdida: 0.0639982505371305
Época 700, Pérdida: 0.042870104694211354
Época 800, Pérdida: 0.03156917617061171
Época 900, Pérdida: 0.024688582592327056

Resultados:
Entrada: [0, 0, 0, 0] -> Pred: 0, Real: 0
Entrada: [0, 0, 0, 1] -> Pred: 1, Real: 1
Entrada: [0, 0, 1, 0] -> Pred: 2, Real: 2
Entrada: [0, 0, 1, 1] -> Pred: 3, Real: 3
Entrada: [0, 1, 0, 0] -> Pred: 4, Real: 4
Entrada: [0, 1, 0, 1] -> Pred: 5, Real: 5
Entrada: [0, 1, 1, 0] -> Pred: 6, Real: 6
Entrada: [0, 1, 1, 1] -> Pred: 7, Real: 7
Entrada: [1, 0, 0, 0] -> Pred: 8, Real: 8
Entrada: [1, 0, 0, 1] -> Pred: 9, Real: 9
Entrada: [1, 0, 1, 0] -> Pred: A, Real: A
Entrada: [1, 0, 1, 1] -> Pred: B, Real: B
Entrada: [1, 1, 0, 0] -> Pred: C, Real: C
Entrada: [1, 1, 0, 1] -> Pr

## Explicación detallada:
1. **Estructura de la Red:**
    - La red tiene 3 capas: entrada (4 neuronas), oculta (8 neuronas) y salida (16 neuronas).
    - Usa ReLU como función de activación en la capa oculta.
    - Usa Softmax en la capa de salida para convertir los valores en probabilidades.
2. **Inicialización:**
    - Los pesos y biases se inicializan con valores aleatorios pequeños entre -0.1 y 0.1.
3. **Forward Pass:**
    - Calcula la salida de cada neurona como la suma ponderada de las entradas, más un bias.
    - Aplica la función de activación correspondiente en cada capa.
4. **Entrenamiento:**
    - Usa backpropagation para ajustar los pesos y minimizar la pérdida.
    - La pérdida se calcula usando entropía cruzada, común para problemas de clasificación.
    - El descenso de gradiente actualiza los pesos en dirección opuesta al gradiente.
5. **Dataset:**
    - Entrena con todas las combinaciones posibles de 4 bits (16 patrones).
    - Cada patrón se clasifica en una de 16 clases (representadas con one-hot encoding).
6. **Prueba:**
    - Evalúa la red con los mismos patrones de entrenamiento (en un caso real usaríamos datos separados).
    - Muestra la predicción y el valor real como dígitos hexadecimales (0-F).
Esta red aprende a mapear cada combinación de 4 bits a su correspondiente clase (de 0 a 15), esencialmente memorizando el conjunto de entrenamiento completo.